# MTI850 - Final Project Template: Big Data Analytics

#### Apache Spark | MLlib | Spark SQL | HDFS | Visualization | Structured Streaming

#### Objective: Apply a complete data processing and machine learning pipeline using Apache Spark on a dataset of your choice.

###### Version 1.0 - 10 September 2025
###### Version 2.0 - 24 September 2026

In [1]:
# -------------------------------------------------------------
# FINAL PROJECT TITLE — BIG DATA ANALYTICS
# -------------------------------------------------------------
# Team
# Student Names: 
# Dataset: 
# Date: 
# -------------------------------------------------------------

### Prerequisite: Start a Spark session

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Big Data Analytics - Final Project") \
    .getOrCreate()


### 1. Dataset Selection & Exploration
##### Load the dataset (can be from local or cloud source)

In [ ]:
# Example: local CSV or from AWS S3 / Google Cloud Storage

# Replace with your actual path
data_path = "home/MTI850/dataset.csv"

df = spark.read.csv(data_path, header=True, inferSchema=True)
df.printSchema()
df.show(5)

### 2. Store Raw Data in HDFS
##### Store the raw dataset into HDFS (you can use CLI or PySpark API)

In [ ]:
# Example (CLI):
# hdfs dfs -mkdir -p /user/<your_username>/final_project
# hdfs dfs -put path/to/your/dataset.csv /user/<your_username>/final_project/

# OR programmatically (PySpark)
df.write.mode('overwrite').csv("hdfs:///home/MTI850/final_project/raw_data", header=True)


### 3. Data Cleaning
##### Handle missing values, remove duplicates, handle incorrect types

In [ ]:
# Check for nulls
from pyspark.sql.functions import col, isnan, when, count

df.select([count(when(col(c).isNull() | isnan(c), c)).alias(c) for c in df.columns]).show()

# Drop duplicates
df_cleaned = df.dropDuplicates()

# Handle nulls or impute values
# Example: df_cleaned = df_cleaned.na.fill({"column_name": "default_value"})

df_cleaned.show(5)

### 4. Data Visualization
##### Convert to Pandas for visualization (small samples only!)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df_sample = df_cleaned.limit(1000).toPandas()

# Example: Visualize correlation matrix
sns.heatmap(df_sample.corr(), annot=True)
plt.title("Feature Correlation Matrix")
plt.show()


### 5. Data Structuring & Feature Engineering
##### Create features, convert types, encode categoricals, etc.

In [ ]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

# Example of string indexing and vector assembly
indexer = StringIndexer(inputCol="category_column", outputCol="category_indexed")
df_indexed = indexer.fit(df_cleaned).transform(df_cleaned)

# Assemble features
assembler = VectorAssembler(
    inputCols=["feature1", "feature2", "category_indexed"], 
    outputCol="features"
)
df_features = assembler.transform(df_indexed)

df_features.select("features", "label_column").show(5)

### 6. Train-Test Split
##### Split the data into training and testing sets

In [ ]:
train_df, test_df = df_features.randomSplit([0.8, 0.2], seed=42)

print(f"Training set count: {train_df.count()}")
print(f"Test set count: {test_df.count()}")

### 7. Model Training (Offline)
##### Train a prediction model using Spark MLlib

In [ ]:
from pyspark.ml.regression import LinearRegression  # or Classification

lr = LinearRegression(featuresCol="features", labelCol="label_column")
lr_model = lr.fit(train_df)

# Model summary
training_summary = lr_model.summary
training_summary.rootMeanSquaredError
training_summary.r2

### 8. Spark SQL — Iterative Querying

In [ ]:
df_features.createOrReplaceTempView("data_view")

# Example SQL query: Average feature1 per category
spark.sql("""
    SELECT category_column, AVG(feature1) as avg_f1
    FROM data_view
    GROUP BY category_column
    ORDER BY avg_f1 DESC
""").show()


### 9. Hyperparameter Tuning

In [ ]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import RegressionEvaluator

paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.01, 0.1, 1.0]) \
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0]) \
    .build()

evaluator = RegressionEvaluator(labelCol="label_column", predictionCol="prediction", metricName="rmse")

cv = CrossValidator(estimator=lr,
                    estimatorParamMaps=paramGrid,
                    evaluator=evaluator,
                    numFolds=3)

cv_model = cv.fit(train_df)

best_model = cv_model.bestModel
print(f"Best Model Params: regParam={best_model._java_obj.getRegParam()}, "
      f"elasticNetParam={best_model._java_obj.getElasticNetParam()}")


### 10. Model Evaluation
##### Evaluate your model on the test data

In [ ]:
predictions = lr_model.transform(test_df)
predictions.select("label_column", "prediction").show(5)

from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    labelCol="label_column", predictionCol="prediction", metricName="rmse")

rmse = evaluator.evaluate(predictions)
print(f"RMSE on test data = {rmse}")


### 11. Save Final Model
##### Save your trained model for future use (offline deployment)

In [ ]:

lr_model.write().overwrite().save("hdfs:///home/MTI850/final_project/model/")

### Final Step: Use Spark Structured Streaming for Model Inference on Streamed Data
Key Concepts for Streaming Integration: 
- Spark Structured Streaming allows you to handle real-time data by defining continuous input-output transformations.
- Model Inference on Streaming Data: After saving the model, you should use it to make predictions on streaming data coming from HDFS files.
- Data Stream Simulation: Simulate streaming data by adding new files to HDFS and Spark will process them in real-time.


### 12. Model Inference on Streaming Data

Load the saved model using PipelineModel.load(). This allows you to reuse the trained model for inference on the streaming data.
- Streaming Data: Define a schema for the streaming data and use spark.readStream to read data from a directory in HDFS where new files are continuously being written.
- Real-time Inference: The transform() function applies the model to each micro-batch of the incoming data, making real-time predictions.
- Output: The writeStream function allows you to write the predictions to the console, but in production, you could write them to other storage solutions like HDFS, Kafka, or a database.


In [ ]:
from pyspark.ml import PipelineModel
from pyspark.sql.functions import col

# Load the saved model
model = PipelineModel.load("hdfs:///home/MTI850/final_project/model/")

# Define the schema for incoming data (this should match your feature structure)
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema = StructType([
    StructField("feature1", IntegerType(), True),
    StructField("feature2", IntegerType(), True),
    StructField("category_column", StringType(), True)
])

# Define the streaming data source (HDFS directory to simulate file stream)
streaming_df = spark.readStream \
    .schema(schema) \
    .csv("hdfs:///homw/MTI850/streaming_data/")

# Apply model for inference (make predictions)
predictions = model.transform(streaming_df)

# Show the predicted results
predictions.select("category_column", "feature1", "feature2", "prediction").writeStream \
    .outputMode("append") \
    .format("console") \
    .start()


### 13. Simulate Data Stream with HDFS
To simulate real-time data, you can add files into the HDFS directory hdfs:///home/MTI850/streaming_data/. Spark Structured Streaming will automatically process them as they arrive.

In [ ]:
# Simulate streaming by adding files to the directory
hdfs dfs -put /path/to/new_data_file.csv hdfs:///home/MTI850/streaming_data/

### 14. Managing Stream Processing
You can monitor the progress of the stream by checking the status:

In [ ]:
# Get the status of the active streaming queries
for query in spark.streams.active:
    print(f"Stream ID: {query.id}, Status: {query.status}, Name: {query.name}")

### 15. Stopping the Stream
To stop the stream, you can either stop the specific query or stop all active queries.

In [ ]:
# Stop all active streaming queries
for query in spark.streams.active:
    query.stop()

## END